WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

Optimize a swelling gel-elastomer rod to match a target CURVATURE profile.

Matching position would demand both the shape and the arc length, and a swelling rod
elongates by an amount set by the material and the bath, so a position target is
over-constrained before the layout is chosen. Matching curvature against the
reference arc coordinate removes that coupling.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.interpolate as spy_interp

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

### Target curvature

In [ ]:
length_scale = 1.0e3  # m to mm . The curve was gen in m. To keep in m, use 1

target_df = pd.read_csv("crops/data/targets/carotid_k_002_left_2.csv", comment="#")
arc_samples = length_scale * target_df["s"].values
target_points = length_scale * target_df[["x", "y", "z"]].values

target_curve_fn = spy_interp.CubicSpline(arc_samples, target_points)

rod_length = float(arc_samples[-1])
num_elems = 150
num_nodes = num_elems + 1
parametric_coords = jnp.linspace(0.0, rod_length, num_nodes)

In [ ]:
target_nodes = jnp.asarray(target_curve_fn(np.asarray(parametric_coords)))
target_kinematics = _mesher.compute_curvature_from_centerline(
  node_coords=target_nodes,
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)
# Non-dimensionalised by rod length so the objective is scale free.
target_curvature = np.asarray(target_kinematics.curvature) * rod_length

station = np.linspace(0.0, 1.0, num_elems)
fig, ax = plt.subplots(figsize=(6.5, 4))
for component, label in enumerate([r"$\kappa_1 L$", r"$\kappa_2 L$", r"$\kappa_3 L$"]):
  ax.plot(station, target_curvature[:, component], label=label)
ax.set_xlabel(r"$\hat{s}/L$")
ax.set_ylabel("normalized target curvature")
ax.legend()
plt.show()

### Geometry

In [ ]:
num_layers_d1 = 12
num_layers_d2 = 5  # > 1 for a space curve
num_fibers = num_layers_d1 * num_layers_d2

aspect_ratio = 18.0
rod_height = rod_length / aspect_ratio
rod_breadth = 0.7 * rod_height
layer_height = rod_height / num_layers_d1
layer_breadth = rod_breadth / num_layers_d2

offsets_d1 = np.linspace(
  -0.5 * (rod_height - layer_height), 0.5 * (rod_height - layer_height), num_layers_d1
)
offsets_d2 = np.linspace(
  -0.5 * (rod_breadth - layer_breadth),
  0.5 * (rod_breadth - layer_breadth),
  num_layers_d2,
)
grid_d1, grid_d2 = np.meshgrid(offsets_d1, offsets_d2, indexing="ij")
offsets_local = np.stack([grid_d1.ravel(), grid_d2.ravel()], axis=-1)

fiber_heights = jnp.full((num_elems, num_fibers), layer_height)
fiber_thicknesses = jnp.full((num_elems, num_fibers), layer_breadth)
fiber_centroid_moi = jnp.zeros((num_elems, num_fibers, 2))
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 0].set(
  layer_height * layer_breadth**3 / 12.0
)
fiber_centroid_moi = fiber_centroid_moi.at[:, :, 1].set(
  layer_breadth * layer_height**3 / 12.0
)

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=parametric_coords,
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), layer_height * layer_breadth),
  fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
  fiber_centroid_moi=fiber_centroid_moi,
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)
print(f"{num_elems * num_fibers} design variables")

### Filter

In [ ]:
arc_centres = 0.5 * (rod.arc_coords[:-1] + rod.arc_coords[1:])
filter_matrix = _utils.create_density_filter(
  coords=arc_centres[:, None],
  cutoff_distance=2.0 * rod_length / num_elems,
)

### Material and environment

In [ ]:
shear_modulus_gel, shear_modulus_elastomer = 1.0e-3, 5.0e-2  # kN/mm^2 = 10^9 N / m^2
chi_gel, chi_elastomer = 0.3, 2.5

solvent = _fe.FlorySolvent(
  bath_temperature=298.0,
  molar_volume=1.8e4,  # #mm^3/mol
  ref_chem_potential=0.0,
)

num_steps = 20

ramp = jnp.linspace(0.0, 1.0, num_steps + 1)[1:] ** 0.03
dry_chem_potential = -1.0e5
wet_chem_potential = -150.0
chem_potentials = dry_chem_potential + (
  wet_chem_potential - dry_chem_potential
) * ramp


### Boundary conditions

In [ ]:
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
boundary_list = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[np.array([0]) for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

# comment this for 3D deformation
# boundary_list += [
#   _bc.DirichletBC(
#     values=(dof, [np.zeros(num_nodes) for _ in range(num_steps)]),
#     node_targets=[np.arange(num_nodes) for _ in range(num_steps)],
#     name=f"planar_{dof}",
#   )
#   for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
# ]

bc = _bc.process_boundary_conditions(rod, boundary_list)

### Solver and problem

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 100,
    "threshold": 1e-8,
    "abs_threshold": 1e-12,
    "lam_min": 0.01,
    "line_search_max_iter": 12,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={
    "solver": _solver.LinearSolvers.SCIPY_SPARSE,
    "rtol": 1e-8,
  },
)
problem = _fe.CosseratProblem(
  rod=rod,
  bc=bc,
  nr_solver=nr_solver,
  solvent=solvent,
  bisection_settings={
    "lower": 1e-6,
    "upper": 1.0 - 1e-6,
    "tol": 1e-6,
    "max_iter": 200,
  },
)

### Objective and constraint

In [ ]:
def compute_objective(design_vars: jnp.ndarray, thresh_beta: float):
  """Mean squared error between achieved and target normalized curvature.

  Both curvatures are evaluated against the same reference arc coordinates, so they
  share a convention and units and are directly comparable once multiplied by the
  rod length. All three Darboux components enter the loss. The third is the twist,
  which a Bishop frame gives as zero for the target, so including it penalises any
  spurious twist the layout would otherwise introduce freely.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` in `[0, 1]`, where one
      is fully gel.
    thresh_beta: Sharpness of the projection pushing the design toward two phases.

  Returns:
    A tuple `(loss, (pseudodensity, u_star))`.
  """
  filtered = filter_matrix @ design_vars.reshape(num_elems, num_fibers)
  pseudodensity = _utils.threshold_filter(filtered, thresh_beta)
  
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=(
      shear_modulus_gel * pseudodensity
      + shear_modulus_elastomer * (1.0 - pseudodensity)
    ),
    flory_parameters=(chi_gel * pseudodensity + chi_elastomer * (1.0 - pseudodensity)),
    chem_potentials=chem_potentials,
  )
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, u_star
  )
  kinematics = _mesher.compute_discrete_cosserat_kinematics(
    node_coords=coords, directors=directors, arc_coords=rod.arc_coords
  )
  error = kinematics.curvature * rod_length - jnp.asarray(target_curvature)
  mse = jnp.mean(jnp.einsum("ec, ec -> e", error, error))
  return jnp.log(mse), (pseudodensity, u_star)


objective_and_grad = jax.jit(
  jax.value_and_grad(compute_objective, argnums=0, has_aux=True)
)


def compute_constraint(
  design_vars: jnp.ndarray, max_grayness: float, thresh_beta: float
):
  """Grayness constraint value and gradient in the shapes MMA expects.

  Grayness is `mean(x * (1 - x))`, zero for a design that is everywhere zero or one
  and maximal at one half. Written as `grayness - max_grayness <= 0`, so a positive
  value is a violation.
  """

  def wrapper(x):
    filtered = filter_matrix @ design_vars.reshape(num_elems, num_fibers)
    projected = _utils.threshold_filter(filtered, thresh_beta)
    return jnp.mean(projected * (1.0 - projected)) - max_grayness

  value, gradient = jax.value_and_grad(wrapper)(design_vars)
  return np.array([[float(value)]]), np.asarray(gradient).reshape(1, -1)

### Optimization

In [ ]:
def draw_state(pseudodensity, u_star, epoch, loss):
  """Deformed shape beside the achieved and target curvature profiles."""
  coords, directors, fiber_coords = (
    _mesher.extract_coords_and_director_from_deformation(rod, u_star)
  )
  achieved = (
    np.asarray(
      _mesher.compute_discrete_cosserat_kinematics(
        node_coords=coords, directors=directors, arc_coords=rod.arc_coords
      ).curvature
    )
    * rod_length
  )

  fig = plt.figure(figsize=(12, 4.5))
  ax_shape = fig.add_subplot(1, 2, 1, projection="3d")
  _viz.plot_rod_fiber_sections(
    fiber_coordinates=np.asarray(fiber_coords),
    directors=np.asarray(directors),
    fiber_heights=np.asarray(fiber_heights),
    fiber_thicknesses=np.asarray(fiber_thicknesses),
    fiber_field=np.asarray(pseudodensity),
    fiber_field_val_range=(0.0, 1.0),
    cmap=_viz.elast_gel_cmap,
    edge_width=0.0,
    ax=ax_shape,
  )
  ax_shape.view_init(elev=25, azim=-60)  # x-y plane
  # ax_shape.view_init(elev=25, azim=-60)   # isometric
  ax_shape.set_title(f"epoch {epoch}   J = {loss:.3E}", fontsize=9)

  ax_curv = fig.add_subplot(1, 2, 2)
  for component, label in enumerate([r"$\kappa_1$", r"$\kappa_2$", r"$\kappa_3$"]):
    (line,) = ax_curv.plot(station, achieved[:, component], lw=1.5, label=label)
    ax_curv.plot(
      station, target_curvature[:, component], "--", lw=1.0, color=line.get_color()
    )
  ax_curv.set_xlabel(r"$\hat{s}/L$")
  ax_curv.set_ylabel(r"normalized curvature $\kappa L$")
  ax_curv.legend(fontsize=8)
  fig.tight_layout()
  plt.show()
  plt.pause(1e-5)

In [ ]:
def run_optimization(init_density, max_iter=201, move_limit=5e-2, save_interval=10):
  """Optimize the two-phase layout against the target curvature with MMA.

  The projection sharpness and the permitted grayness are both annealed with the
  epoch, so the design is free to be gray early and is pushed toward two phases as
  it settles. The objective is rescaled once so MMA's move limits and KKT tolerance
  act on a quantity of order unity.

  The design and displacement fields are stored every `save_interval` epochs and
  once more after the loop, so `history["design"][-1]` and `history["disp"][-1]` are
  always the final state and nothing needs to be re-solved afterwards. They are not
  stored every epoch because each snapshot is a full field.

  Args:
    init_density: Array of shape `(num_elems * num_fibers,)` in `[0, 1]`.
    max_iter: Maximum MMA epochs.
    move_limit: MMA move limit per epoch.
    save_interval: Store and redraw every this many epochs. The state comes from the
      auxiliary output of the objective, so it costs no extra solve.

  Returns:
    A tuple `(mma_state, history)`.
  """
  num_design_var = init_density.shape[0]
  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-6,
    step_tol=1e-6,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=1,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(np.asarray(init_density).reshape(-1, 1), mma_params)
  history = {
    "epoch": [],
    "objective": [],
    "constraint": [],
    "saved_epoch": [],
    "design": [],
    "disp": [],
  }
  obj_scale = 1.0

  while not mma_state.is_converged and mma_state.epoch < max_iter:
    max_grayness = max(0.02, 0.5 - 0.008 * mma_state.epoch)
    thresh_beta = min(64.0, 1.0 + 0.4 * mma_state.epoch)
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    (loss, (pseudodensity, u_star)), grad_obj = objective_and_grad(
      design_vars, thresh_beta
    )
    constraint, grad_constraint = compute_constraint(
      design_vars, max_grayness, thresh_beta
    )
    if mma_state.epoch == 10 or mma_state.epoch == 50:
      obj_scale = 1.0 / jax.lax.stop_gradient(jnp.abs(loss))

    history["epoch"].append(mma_state.epoch)
    history["objective"].append(float(loss))
    history["constraint"].append(float(constraint[0, 0]))
    print(f"epoch {mma_state.epoch:4d}  J {loss:.4E}  gray {constraint[0, 0]:+.4f}")

    if mma_state.epoch % save_interval == 0:
      history["saved_epoch"].append(mma_state.epoch)
      history["design"].append(np.asarray(pseudodensity))
      history["disp"].append(np.asarray(u_star))
      draw_state(pseudodensity, u_star, mma_state.epoch, loss)

    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.array([float(loss) * obj_scale]),
      grad_obj=np.asarray(grad_obj, dtype=np.float64).reshape(-1, 1) * obj_scale,
      cons=constraint,
      grad_cons=grad_constraint,
    )

  history["saved_epoch"].append(mma_state.epoch)
  history["design"].append(np.asarray(pseudodensity))
  history["disp"].append(np.asarray(u_star))
  return mma_state, history

In [ ]:
init_density = 0.5 * jnp.ones((num_elems * num_fibers,))
mma_state, history = run_optimization(init_density)

### Results

In [ ]:
fig, (ax_obj, ax_cons) = plt.subplots(1, 2, figsize=(11, 4))
ax_obj.semilogy(history["epoch"], history["objective"])
ax_obj.set_xlabel("MMA epoch")
ax_obj.set_ylabel("normalized curvature error")
ax_cons.plot(history["epoch"], history["constraint"])
ax_cons.axhline(0.0, color="k", lw=0.5)
ax_cons.set_xlabel("MMA epoch")
ax_cons.set_ylabel("grayness constraint")
fig.tight_layout()
plt.show()

In [ ]:
final_design = history["design"][-1]
final_disp = history["disp"][-1]
draw_state(
  final_design, final_disp, history["saved_epoch"][-1], history["objective"][-1]
)

coords, _, _ = _mesher.extract_coords_and_director_from_deformation(rod, final_disp)
chords = np.linalg.norm(np.diff(np.asarray(coords), axis=0), axis=1)
nodal = np.asarray(final_disp).reshape(num_nodes, rod.num_dofs_per_node)
print(f"deformed arc length / L = {np.sum(chords) / rod_length:.4f}")
print(f"max nodal rotation      = {np.degrees(np.abs(nodal[:, 3:]).max()):.1f} deg")

### Animation

The optimization history already carries the deformed state at every saved epoch, so
the design trajectory can be replayed without re-solving anything.

In [ ]:
import pathlib

import imageio.v2 as imageio

def render_optimization_gif(
  history,
  gif_path="swell_opt_curvature.gif",
  frame_dir="frames/swell_curvature",
  plot_every=10,
  frame_duration=250.0,
  hold_last=6,
  figsize=(15.0, 5.0),
  dpi=150,
):
  """Render front, side, and top views of the optimization history."""

  if plot_every < 1:
    raise ValueError("plot_every must be at least 1")

  frame_dir = pathlib.Path(frame_dir)
  frame_dir.mkdir(parents=True, exist_ok=True)

  loss_by_epoch = dict(zip(history["epoch"], history["objective"]))

  frame_indices = list(range(0, len(history["disp"]), plot_every))
  final_index = len(history["disp"]) - 1
  if frame_indices[-1] != final_index:
    frame_indices.append(final_index)

  states = [
    _mesher.extract_coords_and_director_from_deformation(
      rod, jnp.asarray(history["disp"][index])
    )
    for index in frame_indices
  ]

  every_coord = np.concatenate(
    [
      np.asarray(fiber_coords).reshape(-1, 3)
      for _, _, fiber_coords in states
    ]
  )
  scene_min = every_coord.min(axis=0)
  scene_max = every_coord.max(axis=0)
  mid_point = 0.5 * (scene_min + scene_max)
  half_width = 0.55 * np.max(scene_max - scene_min)

  views = (
    ("Top: XY", 90.0, -90.0),
    ("Front: XZ", 0.0, -90.0),
    ("Side: YZ", 0.0, 0.0),
  )

  frame_paths = []

  for frame_number, (state_index, state) in enumerate(
    zip(frame_indices, states)
  ):
    _, directors, fiber_coords = state
    design = history["design"][state_index]
    epoch = history["saved_epoch"][state_index]
    loss = loss_by_epoch.get(epoch)

    fig, axes = plt.subplots(
      1,
      3,
      figsize=figsize,
      subplot_kw={"projection": "3d"},
    )

    for ax, (view_name, elev, azim) in zip(axes, views):
      _viz.plot_rod_fiber_sections(
        fiber_coordinates=np.asarray(fiber_coords),
        directors=np.asarray(directors),
        fiber_heights=np.asarray(fiber_heights),
        fiber_thicknesses=np.asarray(fiber_thicknesses),
        fiber_field=np.asarray(design),
        fiber_field_val_range=(0.0, 1.0),
        cmap=_viz.elast_gel_cmap,
        edge_width=0.0,
        ax=ax,
      )

      ax.view_init(elev=elev, azim=azim)
      ax.set_xlim3d(mid_point[0] - half_width, mid_point[0] + half_width)
      ax.set_ylim3d(mid_point[1] - half_width, mid_point[1] + half_width)
      ax.set_zlim3d(mid_point[2] - half_width, mid_point[2] + half_width)
      ax.set_box_aspect((1.0, 1.0, 1.0))

      ax.set_xlabel("X displacement")
      ax.set_ylabel("Y displacement")
      ax.set_zlabel("Z displacement")
      ax.set_title(view_name)

    title = f"epoch {epoch}"
    if loss is not None:
      title += f"   J = {loss:.3E}"

    fig.suptitle(title, fontsize=11)
    fig.tight_layout()

    frame_path = frame_dir / f"frame_{frame_number:03d}.png"
    fig.savefig(frame_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    frame_paths.append(frame_path)

  frames = [imageio.imread(path) for path in frame_paths]
  frames += [frames[-1]] * hold_last

  imageio.mimsave(
    gif_path,
    frames,
    duration=frame_duration,
    loop=0,
  )

  return gif_path


gif_path = render_optimization_gif(
  history,
  plot_every=10,
)

print(f"wrote {gif_path} from every 10th saved frame")